# ⚡ MY AI STUDIO — BỘ TĂNG TỐC GPU CLOUD (TESLA T4 16GB)
> **Đường truyền cố định Ngrok vĩnh viễn (Chuẩn như Omni Voice — 100% không bao giờ bị Google ngắt kết nối)**
> 
> 1. Vào menu: **Thời gian chạy** ➔ **Thay đổi loại thời gian chạy** ➔ Chọn **T4 GPU** ➔ Lưu.
> 2. Bấm đúng **1 NÚT PLAY (▶️)** duy nhất ở ô bên dưới.
> 3. Đợi ~1 phút, Colab sẽ kết nối vào tên miền cố định: `https://upturned-evict-geologic.ngrok-free.dev`.
> 4. Quay lại **My AI Studio (Tab Dựng Video ➔ Hoán Đổi Mặt)** bấm **'Kiểm Tra Kết Nối'** là thông ngay lập tức!

In [ ]:
#@title ▶️ KHỞI CHẠY BỘ TĂNG TỐC GPU CLOUD (TURBO TESLA T4 ~35-45 FPS)
#@markdown Tự động cấu hình CUDA 12 và kết nối đường truyền Ngrok/Pinggy ổn định 100%.

NGROK_AUTHTOKEN = "3JYfiFRrXHoFYeMq5hh219lbnyR_2VF5LDb2AJDpiSo1SefAm" #@param {type:"string"}
NGROK_DOMAIN = "upturned-evict-geologic.ngrok-free.dev" #@param {type:"string"}

import os
import sys
import time
import re
import site
import glob
import subprocess

print("=" * 70)
print("⚡ MY AI STUDIO — GPU COMPUTE ACCELERATOR (TESLA T4 TURBO ~40 FPS)")
print("=" * 70)

# 1. Kiểm tra GPU Tesla T4
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        torch.cuda.init()
        _ = torch.zeros(1).cuda()
        print(f"✅ Đã kích hoạt GPU: {gpu_name} (15.3GB VRAM) — Sẵn sàng Turbo!")
    else:
        print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào menu: Thời gian chạy ➔ Thay đổi loại thời gian chạy ➔ Chọn T4 GPU!")
except Exception:
    pass

# 2. Cài đặt thư viện môi trường & Tối ưu CUDA 12
print("\n📦 [1/4] Cài đặt và cấu hình tối ưu CUDA 12 Turbo cho GPU Tesla T4...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart", "opencv-python-headless", "insightface", "onnx", "tqdm", "pyngrok"], check=True)

# Cài đặt các gói CUDA 12 chính chủ từ NVIDIA
print("  ⏳ Cài đặt bộ đệm CUDA 12 & cuDNN 9...", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-cuda-runtime-cu12", "nvidia-cublas-cu12", "nvidia-cudnn-cu12", "nvidia-cufft-cu12", "nvidia-curand-cu12", "nvidia-cuda-nvrtc-cu12"], check=False)

# Cài đặt ONNX Runtime GPU (Tự động thích ứng Python 3.10 - 3.13+)
print("  ⏳ Kích hoạt bộ gia tốc ONNX Runtime GPU (CUDA 12 Turbo)...", flush=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime", "onnxruntime-gpu"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "onnxruntime-gpu"], check=False)
try:
    import onnxruntime as ort
    print(f"  ✅ Đã kích hoạt ONNX Runtime {ort.__version__} (Providers: {ort.get_available_providers()})", flush=True)
except Exception:
    print("  ⏳ Cài đặt gói ONNX Runtime tương thích dự phòng...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"], check=False)
    try:
        import onnxruntime as ort
        print(f"  ✅ Đã kích hoạt ONNX Runtime {ort.__version__}", flush=True)
    except Exception as e:
        print(f"  ⚠️ Cảnh báo nạp ONNX: {e}", flush=True)
# 3. Kỹ thuật Symlink toàn bộ thư viện CUDA 12 vào /usr/lib để Dynamic Linker nạp tức thì
print("  ⏳ Liên kết thư viện CUDA 12 vào hệ thống...", flush=True)
for sp in site.getsitepackages():
    nv_libs = glob.glob(f"{sp}/nvidia/*/lib/*.so*")
    for lib in nv_libs:
        dest = f"/usr/lib/{os.path.basename(lib)}"
        if not os.path.exists(dest):
            try:
                os.symlink(lib, dest)
            except Exception:
                pass
        dest64 = f"/usr/local/cuda/lib64/{os.path.basename(lib)}"
        if os.path.exists("/usr/local/cuda/lib64") and not os.path.exists(dest64):
            try:
                os.symlink(lib, dest64)
            except Exception:
                pass

subprocess.run(["ldconfig"], check=False)

# Thiết lập LD_LIBRARY_PATH cho Subprocess
proc_env = os.environ.copy()
ld_dirs = ["/usr/lib", "/usr/local/cuda/lib64", "/usr/local/cuda-12/lib64"]
for sp in site.getsitepackages():
    nv = os.path.join(sp, 'nvidia')
    if os.path.exists(nv):
        for sub in os.listdir(nv):
            p = os.path.join(nv, sub, 'lib')
            if os.path.isdir(p):
                ld_dirs.append(p)
proc_env["LD_LIBRARY_PATH"] = ":".join(ld_dirs) + ":" + proc_env.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = proc_env["LD_LIBRARY_PATH"]

# 4. Tải mã nguồn Worker
print("\n💾 [2/4] Tải và chuẩn bị động cơ xử lý Face Swap...")
worker_url = "https://raw.githubusercontent.com/nviethiep55-glitch/my-ai-studio-colab/main/worker.py"
subprocess.run(["wget", "-q", "-O", "worker.py", worker_url], check=False)
if not os.path.exists("worker.py") or os.path.getsize("worker.py") < 100:
    subprocess.run(["curl", "-sL", worker_url, "-o", "worker.py"], check=False)

# 5. Khởi chạy GPU Server
print("\n⚡ [3/4] Khởi động GPU Server với CUDA Turbo...")
subprocess.run(["pkill", "-f", "worker.py"], check=False)
subprocess.run(["pkill", "-f", "ngrok"], check=False)
time.sleep(1)

server_proc = subprocess.Popen([sys.executable, "worker.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=proc_env)
time.sleep(2)

# 6. Mở đường truyền kết nối
print("\n🔗 [4/4] Mở đường truyền kết nối về My AI Studio...")
share_url = None
try:
    from pyngrok import ngrok
    ngrok.kill()
    ngrok.set_auth_token(str(NGROK_AUTHTOKEN).strip())
    t_opts = {"addr": 8000, "proto": "http"}
    if NGROK_DOMAIN and str(NGROK_DOMAIN).strip():
        t_opts["domain"] = str(NGROK_DOMAIN).strip()
    tunnel = ngrok.connect(**t_opts)
    share_url = tunnel.public_url.replace("http://", "https://")
    print("\n" + "=" * 70)
    print("🎉 ĐÃ KẾT NỐI ĐƯỜNG TRUYỀN NGROK CỐ ĐỊNH:")
    print(f"\n👉  {share_url}  👈\n")
    print("💡 Đường dẫn này đã được lưu cố định trong My AI Studio!")
    print("   Bạn chỉ cần bấm 'Kiểm Tra Kết Nối' trên Studio là chạy ngay!")
    print("=" * 70 + "\n")
except Exception as ngrok_err:
    print(f"⚠️ Ngrok không khả dụng: {ngrok_err}. Đang mở cổng dự phòng Pinggy...")
    try:
        pinggy_cmd = ["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30", "-p", "443", "-R0:localhost:8000", "a.pinggy.io"]
        p_proc = subprocess.Popen(pinggy_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for _ in range(25):
            line = p_proc.stdout.readline()
            if line:
                match = re.search(r"https://[a-zA-Z0-9-]+\.a\.pinggy\.link", line)
                if match:
                    share_url = match.group(0)
                    print("\n" + "=" * 70)
                    print("🎉 ĐÃ MỞ ĐƯỜNG TRUYỀN DỰ PHÒNG PINGGY THÀNH CÔNG:")
                    print(f"\n👉  {share_url}  👈\n")
                    print("👉 Hãy copy link trên dán vào ô 'URL Colab Worker' bên My AI Studio!")
                    print("=" * 70 + "\n")
                    break
            time.sleep(0.2)
    except Exception as pinggy_e:
        print(f"⚠️ Pinggy lỗi: {pinggy_e}")

try:
    while True:
        line = server_proc.stdout.readline()
        if line:
            print(line, end="", flush=True)
        time.sleep(0.05)
except KeyboardInterrupt:
    print("\n🛑 Đã dừng GPU Worker!")
    server_proc.terminate()
